# 🎯 DTLN Model Training & Evaluation Pipeline
This notebook provides an automated pipeline for:
- 📦 Installing dependencies
- 🏋️ Training DTLN model
- 📊 Evaluating trained model
- 🔄 Converting model to ONNX/TFLite format

**Supported Audio Formats:** WAV, MP3, FLAC, OGG, M4A, AAC, WMA, AIFF, APE

Dataset structure expected:
```
dataset/
├── train/
│   ├── clean/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
│   └── noise/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
├── validation/
│   ├── clean/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
│   └── noise/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
└── test/
    ├── clean/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
    └── noise/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
```

**Note:** All audio files will be automatically converted to 16kHz mono WAV format during training preparation.

In [ ]:
#@title **📦 Install Dependencies & Clone Repository** { display-mode: "form" }

import subprocess
import sys
import os
from pathlib import Path

print("🔧 Installing system and Python dependencies...")
print("=" * 60)

try:
    # Install system dependencies for audio format support
    print("📥 Installing system audio libraries (ffmpeg, libsndfile1)...")
    subprocess.run(
        ['apt-get', 'update', '-qq'],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("✅ System audio libraries installed")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to install system packages: {e}")
    sys.exit(1)

# Clone repository if not exists
repo_dir = Path('/content/DTLN-Retrain')

if not repo_dir.exists():
    print("\n📥 Cloning DTLN-Retrain repository...")
    try:
        subprocess.run(
            ['git', 'clone', 'https://github.com/YOUR_USERNAME/DTLN-Retrain.git'],
            cwd='/content',
            check=True,
            capture_output=True
        )
        print("✅ Repository cloned successfully")
    except subprocess.CalledProcessError as e:
        print("⚠️ Failed to clone repository")
        print("💡 Creating directory structure manually...")
        repo_dir.mkdir(parents=True, exist_ok=True)
        print("❌ Please manually upload DTLN files or check the repository URL")
        sys.exit(1)
else:
    print("\n✅ Repository already exists")

# Change to repository directory
os.chdir(repo_dir)
print(f"📁 Working directory: {os.getcwd()}")

# Check if requirements.txt exists
requirements_file = repo_dir / 'requirements.txt'

if not requirements_file.exists():
    print("\n⚠️ requirements.txt not found, creating default...")
    # Create requirements.txt if it doesn't exist
    requirements_content = """# TensorFlow and Deep Learning
tensorflow==2.10.0

# Audio Processing
soundfile>=0.10.3
librosa>=0.9.0
wavinfo>=1.0.0
pydub>=0.25.1

# Model Conversion
tf2onnx>=1.13.0
onnx>=1.12.0

# Utilities
tqdm>=4.62.0
numpy>=1.21.0
scipy>=1.7.0

# Optional: For audio resampling
resampy>=0.3.1
"""
    with open(requirements_file, 'w') as f:
        f.write(requirements_content)
    print("✅ Created requirements.txt")

# Install Python packages from requirements.txt
print("\n📥 Installing Python packages from requirements.txt...")
print("=" * 60)

try:
    # Use pip install with requirements.txt
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
        check=True,
        capture_output=True,
        text=True
    )
    print("✅ All Python packages installed successfully")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to install Python packages")
    print(f"Error: {e.stderr}")
    print("\n💡 Trying to install packages individually...")
    
    # Fallback: try installing line by line
    with open(requirements_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                try:
                    package = line.split('#')[0].strip()
                    if package:
                        print(f"   Installing {package}...")
                        subprocess.run(
                            [sys.executable, '-m', 'pip', 'install', '-q', package],
                            check=True,
                            stdout=subprocess.DEVNULL
                        )
                except:
                    print(f"   ⚠️ Failed to install {package}, continuing...")
    
    print("✅ Package installation completed with some warnings")

print("\n✅ Environment setup completed!")
print("=" * 60)

In [ ]:
#@title **🛠️ Helper Functions** { display-mode: "form" }

import os
import subprocess
import sys
from pathlib import Path
from typing import List, Optional, Tuple
import json
import librosa
import soundfile as sf
from tqdm import tqdm
import shutil

# Supported audio extensions
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.wma', '.aiff', '.ape', '.opus', '.webm'}

def find_datasets(base_path: Path, data_type: str) -> List[str]:
    """
    Find available datasets in the specified path.
    
    Args:
        base_path: Base path to search (e.g., /content/dataset)
        data_type: 'clean' or 'noise'
        
    Returns:
        List of dataset names
    """
    datasets = set()
    
    for split in ['train', 'validation', 'test']:
        split_path = base_path / split / data_type
        if split_path.exists():
            for item in split_path.iterdir():
                if item.is_dir():
                    datasets.add(item.name)
    
    return sorted(list(datasets))


def count_audio_files(directory: Path) -> int:
    """
    Count audio files in directory.
    
    Args:
        directory: Directory to scan
        
    Returns:
        Number of audio files
    """
    count = 0
    for ext in AUDIO_EXTENSIONS:
        count += len(list(directory.rglob(f'*{ext}')))
    return count


def convert_audio_to_wav(input_path: Path, output_path: Path, target_sr: int = 16000) -> bool:
    """
    Convert any audio format to 16kHz mono WAV format.
    
    Args:
        input_path: Input audio file path
        output_path: Output WAV file path
        target_sr: Target sampling rate (default: 16000)
        
    Returns:
        True if successful
    """
    try:
        # Load audio with librosa (handles multiple formats via ffmpeg)
        audio, sr = librosa.load(input_path, sr=target_sr, mono=True)
        
        # Save as WAV
        output_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(output_path, audio, target_sr)
        
        return True
    except Exception as e:
        print(f"⚠️ Failed to convert {input_path.name}: {e}")
        return False


def prepare_dataset_for_training(base_path: Path, clean_dataset: str, 
                                 noise_dataset: str, output_path: Path,
                                 target_sr: int = 16000) -> Tuple[bool, dict]:
    """
    Prepare dataset by converting all audio files to 16kHz mono WAV format.
    
    Args:
        base_path: Base dataset path
        clean_dataset: Clean dataset name
        noise_dataset: Noise dataset name
        output_path: Output path for converted dataset
        target_sr: Target sampling rate
        
    Returns:
        Tuple of (success, stats_dict)
    """
    print("\n🔄 Preparing dataset for training...")
    print("📝 Converting all audio files to 16kHz mono WAV format")
    print("=" * 60)
    
    stats = {
        'train': {'clean': 0, 'noise': 0},
        'validation': {'clean': 0, 'noise': 0},
        'test': {'clean': 0, 'noise': 0}
    }
    
    for split in ['train', 'validation', 'test']:
        print(f"\n📁 Processing {split} split...")
        
        # Process clean files
        clean_source = base_path / split / 'clean' / clean_dataset
        clean_target = output_path / split / 'clean' / clean_dataset
        
        if clean_source.exists():
            print(f"   🧹 Converting clean audio files...")
            audio_files = []
            for ext in AUDIO_EXTENSIONS:
                audio_files.extend(list(clean_source.rglob(f'*{ext}')))
            
            for audio_file in tqdm(audio_files, desc=f"   Clean {split}"):
                # Preserve directory structure
                rel_path = audio_file.relative_to(clean_source)
                output_file = clean_target / rel_path.with_suffix('.wav')
                
                if convert_audio_to_wav(audio_file, output_file, target_sr):
                    stats[split]['clean'] += 1
        
        # Process noise files
        noise_source = base_path / split / 'noise' / noise_dataset
        noise_target = output_path / split / 'noise' / noise_dataset
        
        if noise_source.exists():
            print(f"   🔊 Converting noise audio files...")
            audio_files = []
            for ext in AUDIO_EXTENSIONS:
                audio_files.extend(list(noise_source.rglob(f'*{ext}')))
            
            for audio_file in tqdm(audio_files, desc=f"   Noise {split}"):
                # Preserve directory structure
                rel_path = audio_file.relative_to(noise_source)
                output_file = noise_target / rel_path.with_suffix('.wav')
                
                if convert_audio_to_wav(audio_file, output_file, target_sr):
                    stats[split]['noise'] += 1
    
    print("\n✅ Dataset preparation completed!")
    print("\n📊 Conversion statistics:")
    for split, counts in stats.items():
        print(f"   {split.capitalize()}:")
        print(f"      Clean: {counts['clean']} files")
        print(f"      Noise: {counts['noise']} files")
    
    return True, stats


def validate_dataset_structure(base_path: Path, clean_dataset: str, 
                               noise_dataset: str) -> Tuple[bool, str, dict]:
    """
    Validate that the dataset structure is correct.
    
    Args:
        base_path: Base dataset path
        clean_dataset: Clean dataset name
        noise_dataset: Noise dataset name
        
    Returns:
        Tuple of (is_valid, error_message, file_counts)
    """
    required_splits = ['train', 'validation']
    file_counts = {}
    
    for split in required_splits:
        clean_path = base_path / split / 'clean' / clean_dataset
        noise_path = base_path / split / 'noise' / noise_dataset
        
        if not clean_path.exists():
            return False, f"Missing: {clean_path}", {}
        
        if not noise_path.exists():
            return False, f"Missing: {noise_path}", {}
        
        # Count audio files of any supported format
        clean_count = count_audio_files(clean_path)
        noise_count = count_audio_files(noise_path)
        
        if clean_count == 0:
            return False, f"No audio files in {clean_path}", {}
        
        if noise_count == 0:
            return False, f"No audio files in {noise_path}", {}
        
        file_counts[split] = {'clean': clean_count, 'noise': noise_count}
    
    return True, "", file_counts


def prepare_mixed_dataset(base_path: Path, clean_dataset: str, 
                         noise_dataset: str, output_path: Path) -> bool:
    """
    Create mixed dataset by combining clean speech with noise.
    Simple implementation: uses noise files directly as "noisy" versions.
    For actual mixing, you would need to implement SNR-based mixing.
    
    Args:
        base_path: Base dataset path
        clean_dataset: Clean dataset name  
        noise_dataset: Noise dataset name
        output_path: Output path for mixed dataset
        
    Returns:
        True if successful
    """
    print("\n🔀 Preparing mixed dataset...")
    print("💡 Using noise files as noisy versions (simplified)")
    print("=" * 60)
    
    for split in ['train', 'validation']:
        noise_path = base_path / split / 'noise' / noise_dataset
        output_split_path = output_path / split
        output_split_path.mkdir(parents=True, exist_ok=True)
        
        # Create symlink or copy noise files
        if not (output_split_path / noise_dataset).exists():
            try:
                # Try symlink first (faster)
                (output_split_path / noise_dataset).symlink_to(
                    noise_path, target_is_directory=True
                )
                print(f"✅ Linked {split} noisy data")
            except:
                # Fallback to copy
                import shutil
                shutil.copytree(noise_path, output_split_path / noise_dataset)
                print(f"✅ Copied {split} noisy data")
    
    print("✅ Mixed dataset prepared")
    return True


def run_training(train_mix_path: str, train_speech_path: str,
                val_mix_path: str, val_speech_path: str,
                run_name: str, gpu: str = '0') -> bool:
    """
    Run DTLN training.
    
    Returns:
        True if successful
    """
    print("\n🏋️ Starting DTLN Training...")
    print("=" * 60)
    
    cmd = [
        sys.executable, 'run_training.py',
        '--train_mix', train_mix_path,
        '--train_speech', train_speech_path,
        '--val_mix', val_mix_path,
        '--val_speech', val_speech_path,
        '--run_name', run_name,
        '--gpu', gpu
    ]
    
    print(f"🔧 Command: {' '.join(cmd)}")
    print("")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        for line in iter(process.stdout.readline, ''):
            if line:
                print(line.rstrip())
                sys.stdout.flush()
        
        process.wait()
        
        if process.returncode == 0:
            print("\n✅ Training completed successfully!")
            return True
        else:
            print(f"\n❌ Training failed with code {process.returncode}")
            return False
            
    except Exception as e:
        print(f"\n❌ Training error: {e}")
        return False


def run_evaluation(input_folder: str, output_folder: str, 
                  model_path: str) -> bool:
    """
    Run DTLN evaluation on test set.
    
    Returns:
        True if successful
    """
    print("\n📊 Starting DTLN Evaluation...")
    print("=" * 60)
    
    cmd = [
        sys.executable, 'run_evaluation.py',
        '--in_folder', input_folder,
        '--out_folder', output_folder,
        '--model', model_path
    ]
    
    print(f"🔧 Command: {' '.join(cmd)}")
    print("")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        for line in iter(process.stdout.readline, ''):
            if line:
                print(line.rstrip())
                sys.stdout.flush()
        
        process.wait()
        
        if process.returncode == 0:
            print("\n✅ Evaluation completed successfully!")
            return True
        else:
            print(f"\n❌ Evaluation failed with code {process.returncode}")
            return False
            
    except Exception as e:
        print(f"\n❌ Evaluation error: {e}")
        return False


def run_conversion(weights_file: str, target_name: str, 
                  format: str = 'onnx') -> bool:
    """
    Convert trained model to ONNX or TFLite format.
    
    Args:
        weights_file: Path to .h5 weights
        target_name: Target name (without extension)
        format: 'onnx' or 'tflite'
        
    Returns:
        True if successful
    """
    print(f"\n🔄 Converting model to {format.upper()}...")
    print("=" * 60)
    
    if format == 'onnx':
        cmd = [
            sys.executable, 'convert_weights_to_onnx.py',
            '--weights_file', weights_file,
            '--target_folder', target_name
        ]
    else:
        print("❌ TFLite conversion not yet implemented in this notebook")
        print("💡 Use DTLN_model.create_tf_lite_model() method directly")
        return False
    
    print(f"🔧 Command: {' '.join(cmd)}")
    print("")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        for line in iter(process.stdout.readline, ''):
            if line:
                print(line.rstrip())
                sys.stdout.flush()
        
        process.wait()
        
        if process.returncode == 0:
            print(f"\n✅ Conversion to {format.upper()} completed!")
            return True
        else:
            print(f"\n❌ Conversion failed with code {process.returncode}")
            return False
            
    except Exception as e:
        print(f"\n❌ Conversion error: {e}")
        return False


print("✅ All helper functions loaded successfully!")
print(f"📋 Supported audio formats: {', '.join(sorted(AUDIO_EXTENSIONS))}")

In [ ]:
#@title **🚀 Configure & Execute Pipeline** { display-mode: "form" }

#@markdown ---
#@markdown ### 📁 Dataset Configuration

dataset_base_path = "/content/dataset" #@param {type:"string"}

#@markdown Select your datasets (must match folder names in dataset structure):
clean_dataset_name = "" #@param {type:"string"}
noise_dataset_name = "" #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Pipeline Stages

run_training_stage = True #@param {type:"boolean"}
run_evaluation_stage = False #@param {type:"boolean"}
run_conversion_stage = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🎵 Audio Processing Options

convert_to_wav = True #@param {type:"boolean"}
#@markdown Convert all audio formats to 16kHz mono WAV (recommended for training)

target_sampling_rate = 16000 #@param {type:"integer"}
#@markdown Target sampling rate for audio conversion (Hz)

#@markdown ---
#@markdown ### 🏋️ Training Configuration

training_run_name = "DTLN_model" #@param {type:"string"}
gpu_device = "0" #@param {type:"string"}

#@markdown ---
#@markdown ### 📊 Evaluation Configuration

#@markdown Path to test dataset (will use noise dataset from test split)
evaluation_output_folder = "/content/enhanced_audio" #@param {type:"string"}

#@markdown Path to trained model weights (.h5 file)
model_weights_path = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🔄 Conversion Configuration

conversion_format = "onnx" #@param ["onnx", "tflite"]
conversion_target_name = "/content/dtln_converted" #@param {type:"string"}

#@markdown ---

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("=" * 60)
print("🎯 DTLN PIPELINE EXECUTION")
print("=" * 60)

base_path = Path(dataset_base_path)

# Validate base dataset path
if not base_path.exists():
    print(f"\n❌ Error: Dataset path does not exist: {base_path}")
    print("💡 Please run the dataset pipeline first to create the dataset structure")
    sys.exit(1)

# ============================================================================
# STAGE 0: AUDIO CONVERSION (if needed)
# ============================================================================

converted_base_path = base_path
if convert_to_wav and run_training_stage:
    print("\n" + "=" * 60)
    print("🎵 STAGE 0: AUDIO FORMAT CONVERSION")
    print("=" * 60)
    
    # Validate inputs
    if not clean_dataset_name or not noise_dataset_name:
        print("\n❌ Error: Dataset names are required!")
        print("💡 Please fill in 'clean_dataset_name' and 'noise_dataset_name'")
        sys.exit(1)
    
    # Check available datasets
    print("\n🔍 Scanning for available datasets...")
    clean_datasets = find_datasets(base_path, 'clean')
    noise_datasets = find_datasets(base_path, 'noise')
    
    print(f"\n📋 Available clean datasets: {', '.join(clean_datasets) if clean_datasets else 'None'}")
    print(f"📋 Available noise datasets: {', '.join(noise_datasets) if noise_datasets else 'None'}")
    
    if clean_dataset_name not in clean_datasets:
        print(f"\n❌ Clean dataset '{clean_dataset_name}' not found!")
        print(f"💡 Available options: {', '.join(clean_datasets)}")
        sys.exit(1)
    
    if noise_dataset_name not in noise_datasets:
        print(f"\n❌ Noise dataset '{noise_dataset_name}' not found!")
        print(f"💡 Available options: {', '.join(noise_datasets)}")
        sys.exit(1)
    
    # Validate dataset structure and count files
    print("\n🔍 Validating dataset structure...")
    is_valid, error_msg, file_counts = validate_dataset_structure(
        base_path, clean_dataset_name, noise_dataset_name
    )
    
    if not is_valid:
        print(f"\n❌ Dataset validation failed: {error_msg}")
        sys.exit(1)
    
    print("✅ Dataset structure is valid")
    print("\n📊 Original dataset files:")
    for split, counts in file_counts.items():
        print(f"   {split.capitalize()}:")
        print(f"      Clean: {counts['clean']} files")
        print(f"      Noise: {counts['noise']} files")
    
    # Prepare converted dataset
    converted_base_path = Path('/content/dataset_converted')
    
    success, stats = prepare_dataset_for_training(
        base_path=base_path,
        clean_dataset=clean_dataset_name,
        noise_dataset=noise_dataset_name,
        output_path=converted_base_path,
        target_sr=target_sampling_rate
    )
    
    if not success:
        print("\n❌ Audio conversion failed!")
        sys.exit(1)
    
    print(f"\n✅ STAGE 0 COMPLETED: Audio converted to {target_sampling_rate}Hz mono WAV")

# ============================================================================
# STAGE 1: TRAINING
# ============================================================================

if run_training_stage:
    print("\n" + "=" * 60)
    print("🏋️ STAGE 1: TRAINING")
    print("=" * 60)
    
    # Validate inputs (if not already validated in conversion stage)
    if not convert_to_wav:
        if not clean_dataset_name or not noise_dataset_name:
            print("\n❌ Error: Dataset names are required!")
            print("💡 Please fill in 'clean_dataset_name' and 'noise_dataset_name'")
            sys.exit(1)
        
        # Check available datasets
        print("\n🔍 Scanning for available datasets...")
        clean_datasets = find_datasets(converted_base_path, 'clean')
        noise_datasets = find_datasets(converted_base_path, 'noise')
        
        print(f"\n📋 Available clean datasets: {', '.join(clean_datasets) if clean_datasets else 'None'}")
        print(f"📋 Available noise datasets: {', '.join(noise_datasets) if noise_datasets else 'None'}")
        
        if clean_dataset_name not in clean_datasets:
            print(f"\n❌ Clean dataset '{clean_dataset_name}' not found!")
            print(f"💡 Available options: {', '.join(clean_datasets)}")
            sys.exit(1)
        
        if noise_dataset_name not in noise_datasets:
            print(f"\n❌ Noise dataset '{noise_dataset_name}' not found!")
            print(f"💡 Available options: {', '.join(noise_datasets)}")
            sys.exit(1)
        
        # Validate dataset structure
        print("\n🔍 Validating dataset structure...")
        is_valid, error_msg, file_counts = validate_dataset_structure(
            converted_base_path, clean_dataset_name, noise_dataset_name
        )
        
        if not is_valid:
            print(f"\n❌ Dataset validation failed: {error_msg}")
            sys.exit(1)
        
        print("✅ Dataset structure is valid")
    
    # Prepare paths (use converted dataset if available)
    train_mix_path = str(converted_base_path / 'train' / 'noise' / noise_dataset_name)
    train_speech_path = str(converted_base_path / 'train' / 'clean' / clean_dataset_name)
    val_mix_path = str(converted_base_path / 'validation' / 'noise' / noise_dataset_name)
    val_speech_path = str(converted_base_path / 'validation' / 'clean' / clean_dataset_name)
    
    print(f"\n📁 Training paths:")
    print(f"   Noisy (mix): {train_mix_path}")
    print(f"   Clean: {train_speech_path}")
    print(f"   Val noisy: {val_mix_path}")
    print(f"   Val clean: {val_speech_path}")
    
    # Run training
    success = run_training(
        train_mix_path=train_mix_path,
        train_speech_path=train_speech_path,
        val_mix_path=val_mix_path,
        val_speech_path=val_speech_path,
        run_name=training_run_name,
        gpu=gpu_device
    )
    
    if success:
        model_save_path = Path(f'./models_{training_run_name}')
        weights_file = model_save_path / f'{training_run_name}.h5'
        print(f"\n📦 Model saved to: {weights_file}")
        print(f"📊 Training logs: {model_save_path / f'training_{training_run_name}.log'}")
    else:
        print("\n❌ Training failed!")
        sys.exit(1)

else:
    print("\n⏭️ STAGE 1: SKIPPED (run_training_stage = False)")

# ============================================================================
# STAGE 2: EVALUATION
# ============================================================================

if run_evaluation_stage:
    print("\n" + "=" * 60)
    print("📊 STAGE 2: EVALUATION")
    print("=" * 60)
    
    # Determine model weights path
    if not model_weights_path:
        # Try to use model from training stage
        if run_training_stage:
            model_weights_path = str(Path(f'./models_{training_run_name}') / f'{training_run_name}.h5')
            print(f"💡 Using model from training: {model_weights_path}")
        else:
            print("\n❌ Error: Model weights path is required!")
            print("💡 Please specify 'model_weights_path' or run training stage first")
            sys.exit(1)
    
    # Check if model exists
    if not Path(model_weights_path).exists():
        print(f"\n❌ Model weights not found: {model_weights_path}")
        sys.exit(1)
    
    # Use test set noise data as input (from converted dataset if available)
    test_input_path = str(converted_base_path / 'test' / 'noise' / noise_dataset_name)
    
    if not Path(test_input_path).exists():
        print(f"\n❌ Test dataset not found: {test_input_path}")
        print("💡 Make sure your dataset has a 'test' split")
        sys.exit(1)
    
    print(f"\n📁 Evaluation paths:")
    print(f"   Input (noisy): {test_input_path}")
    print(f"   Output (enhanced): {evaluation_output_folder}")
    print(f"   Model: {model_weights_path}")
    
    # Run evaluation
    success = run_evaluation(
        input_folder=test_input_path,
        output_folder=evaluation_output_folder,
        model_path=model_weights_path
    )
    
    if success:
        print(f"\n✅ Enhanced audio saved to: {evaluation_output_folder}")
    else:
        print("\n❌ Evaluation failed!")
        sys.exit(1)

else:
    print("\n⏭️ STAGE 2: SKIPPED (run_evaluation_stage = False)")

# ============================================================================
# STAGE 3: CONVERSION
# ============================================================================

if run_conversion_stage:
    print("\n" + "=" * 60)
    print("🔄 STAGE 3: MODEL CONVERSION")
    print("=" * 60)
    
    # Determine model weights path
    if not model_weights_path:
        if run_training_stage:
            model_weights_path = str(Path(f'./models_{training_run_name}') / f'{training_run_name}.h5')
            print(f"💡 Using model from training: {model_weights_path}")
        else:
            print("\n❌ Error: Model weights path is required!")
            print("💡 Please specify 'model_weights_path' or run training stage first")
            sys.exit(1)
    
    # Check if model exists
    if not Path(model_weights_path).exists():
        print(f"\n❌ Model weights not found: {model_weights_path}")
        sys.exit(1)
    
    print(f"\n📁 Conversion configuration:")
    print(f"   Input model: {model_weights_path}")
    print(f"   Output format: {conversion_format.upper()}")
    print(f"   Target name: {conversion_target_name}")
    
    # Run conversion
    success = run_conversion(
        weights_file=model_weights_path,
        target_name=conversion_target_name,
        format=conversion_format
    )
    
    if success:
        if conversion_format == 'onnx':
            print(f"\n✅ ONNX models saved:")
            print(f"   • {conversion_target_name}_1.onnx")
            print(f"   • {conversion_target_name}_2.onnx")
    else:
        print("\n❌ Conversion failed!")
        sys.exit(1)

else:
    print("\n⏭️ STAGE 3: SKIPPED (run_conversion_stage = False)")

# ============================================================================
# CLEANUP
# ============================================================================

if convert_to_wav and converted_base_path != base_path:
    print("\n" + "=" * 60)
    print("🧹 CLEANUP")
    print("=" * 60)
    
    try:
        # Optionally clean up converted files after training
        cleanup_converted = False  # Set to True if you want to clean up
        
        if cleanup_converted and converted_base_path.exists():
            print("🗑️ Cleaning up converted audio files...")
            shutil.rmtree(converted_base_path)
            print("✅ Converted files cleaned up")
        else:
            print(f"💾 Converted files kept at: {converted_base_path}")
            print("💡 You can manually delete this folder if needed")
    except Exception as e:
        print(f"⚠️ Warning: Could not clean converted files: {e}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETED!")
print("=" * 60)

executed_stages = []
if convert_to_wav and run_training_stage:
    executed_stages.append("Audio Conversion")
if run_training_stage:
    executed_stages.append("Training")
if run_evaluation_stage:
    executed_stages.append("Evaluation")
if run_conversion_stage:
    executed_stages.append("Model Conversion")

print(f"\n✅ Executed stages: {', '.join(executed_stages) if executed_stages else 'None'}")

if convert_to_wav and run_training_stage:
    print(f"\n🎵 Audio files converted to {target_sampling_rate}Hz mono WAV")
    print(f"   Location: {converted_base_path}")

if run_training_stage:
    print(f"\n📦 Trained model: ./models_{training_run_name}/{training_run_name}.h5")

if run_evaluation_stage:
    print(f"📊 Enhanced audio: {evaluation_output_folder}")

if run_conversion_stage:
    print(f"🔄 Converted model: {conversion_target_name}_{{1,2}}.{conversion_format}")

print("\n💡 Next steps:")
if run_training_stage and not run_evaluation_stage:
    print("   • Enable 'run_evaluation_stage' to test the model")
elif run_evaluation_stage and not run_conversion_stage:
    print("   • Enable 'run_conversion_stage' to export the model")
elif not any([run_training_stage, run_evaluation_stage, run_conversion_stage]):
    print("   • Enable at least one stage to run the pipeline")
else:
    print("   • Your model is ready for deployment!")

print(f"\n📋 Supported audio formats: WAV, MP3, FLAC, OGG, M4A, AAC, WMA, AIFF, APE, OPUS, WEBM")